# Figure 3 Supplementary — Acquisition Robustness (consolidated)

One notebook pulling together every Figure-3 robustness analysis under a single, consistent
statistics layer. **The data layer (boxes, null distributions, percent-replicating values, depth
curves) is reproduced verbatim from the source notebooks — only the significance-test layer
changes.**

Panels: **A** z-plane subsampling (v1, c=52) · **A2** z-sampling density (v1/14z/44z) ·
**A3** spheroid size · **A4** clearing · **A5** magnification ·
**B** intensity decay/bleaching · **C** cell detection vs depth.

Statistics (A-panels; **no omnibus, no mAP**):
1. **Within-condition** (replicate vs null) — two-sided **Mann–Whitney U** comparing each compound's
   median replicate correlation to that condition's non-replicate (null) distribution; **raw/nominal p,
   no correction** (stars; each condition is one question).
2. **Post-hoc, all-vs-all** on the **gap-corrected** paired Δ (Δ = baseline − comparison), with the
   **bootstrap CI over compounds as the primary verdict for every pair**: designated **equivalence**
   pairs use a **90% CI vs ±0.10** (≡/inconclusive/≠); every other (**difference**) pair uses a
   **95% CI** (excludes 0 = ≠). The gap-corrected **signed-rank p is reported as a nominal, uncorrected
   companion**, not the decision. All pairs shown (exploratory); confounded pairs flagged. **No BH.**
3. **Depth curves (B/C)** — cluster-permutation (B: BH across 5 channels; C: BH across condition-pairs).
Equivalence margin **Δ = 0.10** (replicate Pearson r).

Sources: `spher_colo52_v1/.../3_PercentReplicating_certain_slices.ipynb` (A);
`spher_colo52_v3/.../clearing_comparison_stats.ipynb` (A2/A4/A5, B, C).

In [ ]:
# --- repo path bootstrap (added by the port) ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, external, require)
from utils.panels import save_panel

# ===== Setup, paths, RNG, and ported data-layer utilities =====
import os, re, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns; sns.set_style('white')
import random
from scipy.stats import mannwhitneyu, wilcoxon
from scipy.stats import t as _tdist
from matplotlib.patches import Patch

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
DPI = 300
rng_global = np.random.RandomState(123)   # parity with source notebooks

V3SEC      = profiles("exp3_clearing_mag_z", "sections")
V3_SLICE   = features("exp3_clearing_mag_z", "150526", "SingleSlice")
V3_SC      = features("exp3_clearing_mag_z", "150526", "SingleCell", "HCT116.parquet")
V1_AGG     = profiles("exp1_main", "selected_data_aggregates_HCT116.parquet")
V1_SLICE   = features("exp1_main", "011225", "SingleSlice")
V1_SC      = features("exp1_main", "011225", "SingleCell", "HCT116.parquet")
V1_SLICES  = profiles("exp1_main", "slices")
OUT        = figdir("SupplFig3")
# Panels g/h aggregate the 19.5 GB feature dumps; the aggregation they plot is
# committed here so the panels regenerate without the dumps.
CACHE_3G   = analysis_input("3_SupplFigure3/data/suppl3g_bleaching_perwell.csv")
CACHE_3H   = analysis_input("3_SupplFigure3/data/suppl3h_detection_perwell.csv")
os.makedirs(OUT, exist_ok=True)

# image_id -> condition (from SESSION_SUMMARY_2026-05-15.md)
IMG2COND = {8787:'nyquist', 8804:'nyquist', 8780:'double_dens', 8799:'double_dens',
            8783:'37C', 8802:'37C', 8793:'noclear', 8795:'noclear',
            8789:'20_40', 8791:'20_40', 8797:'20_40',
            8827:'10x', 8831:'10x', 8825:'40x', 8829:'40x'}
CHANNELS = ['HOECHST', 'SYTO', 'CONC', 'PHAandWGA', 'MITO']
LBL = {'20_40':'20/40', 'double_dens':'double dens', 'noclear':'no clear',
       'nyquist_all':'nyquist (44z)', 'nyquist_14planes':'nyquist (14z)', '37C':'37 °C',
       '10x':'10×', '40x':'40×', 'v1':'v1 (orig)'}

# v3 short -> v1 cmpdname; matched 20_40 dose (uM), SN-38 corrected to 1 uM
V1NAME = {'abemaciclib':'abemaciclib (LY2835219)',
          'binimetinib':'Binimetinib (MEK162, ARRY-162, ARRY-438162)',
          'fluorouracil':'Fluorouracil (5-Fluoracil, 5-FU)', 'gemcitabine':'Gemcitabine',
          'sn-38':'SN-38', 'trifluridine':'Trifluridine'}
V1DOSE = {'abemaciclib':10.0, 'binimetinib':10.0, 'fluorouracil':10.0,
          'gemcitabine':1.0, 'sn-38':1.0, 'trifluridine':10.0}

def featurecols(df):
    return [c for c in df.columns if 'Metadata' not in c and pd.api.types.is_numeric_dtype(df[c])]
def shared_features(*dfs):
    return sorted(set.intersection(*[set(featurecols(d)) for d in dfs]))
def load_v3(cond):
    df = pd.read_parquet(f'{V3SEC}/selected_{cond}.parquet').dropna(axis='columns', how='all')
    return df[df['Metadata_pert_type'] == 'trt'].copy()
def load_v1_agg():
    d = pd.read_parquet(V1_AGG)
    assert (d['Metadata_cell_line'] == 'HCT116').all(), 'v1 aggregates contain non-HCT116 rows!'
    return d[d['Metadata_cell_line'] == 'HCT116'].copy()

def matched_null(C, idxby, n_samples, rng):
    # Fair null = median pairwise r of a NON-replicate group: one well per compound (Cimini-style).
    ucmps = list(idxby); k = len(ucmps); ii, jj = np.triu_indices(k, 1)
    sel = np.stack([rng.choice(idxby[c], size=n_samples) for c in ucmps], axis=1)
    return np.nanmedian(C[sel[:, ii], sel[:, jj]], axis=1)
def matched_null_df(d, fs, cmpd_col, n_samples=2000, seed=0):
    X = d[fs].values; lab = d[cmpd_col].values
    C = np.corrcoef(X); np.fill_diagonal(C, np.nan)
    idxby = {c: np.where(lab == c)[0] for c in np.unique(lab)}
    return matched_null(C, idxby, n_samples, np.random.default_rng(seed))

# --- ported percent-replicating utils (for Panel A, v1 slices) ---
def get_featurecols(df): return [c for c in df.columns if not 'Metadata' in c]
def get_featuredata(df): return df[get_featurecols(df)]
def corr_between_replicates(df, group_by_feature):
    replicate_corr = []
    for name, group in df.groupby(group_by_feature):
        gf = get_featuredata(group); corr = np.corrcoef(gf)
        if len(gf) == 1: replicate_corr.append(np.nan)
        else:
            np.fill_diagonal(corr, np.nan); replicate_corr.append(np.nanmedian(corr))
    return replicate_corr
def corr_between_non_replicates(df, n_samples, n_replicates, metadata_compound_name):
    df = df.reset_index(drop=True); null_corr = []; random.seed(42)
    while len(null_corr) < n_samples:
        compounds = random.choices([_ for _ in range(len(df))], k=n_replicates)
        sample = df.loc[compounds].copy()
        if len(sample[metadata_compound_name].unique()) == n_replicates:
            sf = get_featuredata(sample); corr = np.corrcoef(sf)
            np.fill_diagonal(corr, np.nan); null_corr.append(np.nanmedian(corr))
    return null_corr
def process_correlation_data(df, perturbation, cmpd_short_name, cmpd_conc):
    data = get_featuredata(df).groupby(df[perturbation]).mean()
    correlations = corr_between_replicates(df, perturbation)
    cd = pd.DataFrame({perturbation: data.index, 'corr': correlations})
    split = cd[perturbation].str.rsplit('_', n=1, expand=True)   # rsplit fix
    cd[cmpd_short_name] = split[0]; cd[cmpd_conc] = split[1]
    return cd
print('setup ok')

In [ ]:
# ===== Depth-curve helpers (ported verbatim from clearing_comparison_stats.ipynb) =====
def _bh_arr(p):
    p = np.asarray(p, float); n = len(p)
    if n == 0: return p
    o = np.argsort(p); r = np.empty(n, int); r[o] = np.arange(1, n + 1)
    adj = p * n / r; s = adj[o][::-1]; s = np.minimum.accumulate(s)[::-1]
    out = np.empty(n); out[o] = np.clip(s, 0, 1); return out
def _stars(p):
    if np.isnan(p): return 'n.d.'
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
def _runs(mask):
    runs = []; i = 0; n = len(mask)
    while i < n:
        if mask[i]:
            j = i
            while j < n and mask[j]: j += 1
            runs.append((i, j)); i = j
        else: i += 1
    return runs
def _tstat(A, B):
    ma, mb = A.mean(0), B.mean(0); va, vb = A.var(0, ddof=1), B.var(0, ddof=1)
    se = np.sqrt(va / len(A) + vb / len(B))
    with np.errstate(invalid='ignore', divide='ignore'):
        t = (ma - mb) / se
    return np.where(se > 0, t, 0.0)
def cluster_perm(A, B, grid, rng, nperm=2000, thr_p=0.05):
    A = np.asarray(A, float); B = np.asarray(B, float); na, nb = len(A), len(B)
    tcrit = _tdist.ppf(1 - thr_p / 2, na + nb - 2); tobs = _tstat(A, B)
    obs = [(i, j, float(np.sum(tobs[i:j]))) for (i, j) in _runs(np.abs(tobs) > tcrit)]
    pooled = np.vstack([A, B]); nullmax = np.empty(nperm)
    for k in range(nperm):
        idx = rng.permutation(na + nb); t = _tstat(pooled[idx[:na]], pooled[idx[na:]])
        nullmax[k] = max([abs(np.sum(t[i:j])) for (i, j) in _runs(np.abs(t) > tcrit)], default=0.0)
    bands = [(grid[i], grid[j - 1], (1 + np.sum(nullmax >= abs(mass))) / (nperm + 1)) for (i, j, mass) in obs]
    return bands, (min([b[2] for b in bands], default=1.0))
def _sig_bands(bands): return [(s, e, p) for (s, e, p) in bands if p < 0.05]
def _ci_diff(a, b, rng, nboot=2000):
    a = np.asarray(a, float); b = np.asarray(b, float)
    d = np.array([np.median(a[rng.integers(0, len(a), len(a))]) - np.median(b[rng.integers(0, len(b), len(b))]) for _ in range(nboot)])
    return float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))
def _boot_ci(vals, rng, stat, nboot=2000):
    vals = np.asarray(vals, float); vals = vals[~np.isnan(vals)]
    if len(vals) == 0: return np.nan, np.nan, np.nan
    bs = stat(vals[rng.integers(0, len(vals), size=(nboot, len(vals)))], axis=1)
    return float(stat(vals)), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))
print('depth helpers ok')

In [ ]:
# ===== Corrected-statistics layer (HONEST: within = replicate-vs-null MWU; between = gap-Δ CI; no omnibus, no mAP, no BH) =====

# fname used by panel_analysis -> paper panel. Panels g and h (bleaching, detection
# depth) are saved inline further down and need the feature dumps; see PORT_TRIAGE.
FNAME_PANEL = {
    "A_zplane_subsampling": "SupplFig3a",
    "A2_zdensity":          "SupplFig3b",
    "A3_spheroid_size":     "SupplFig3c",
    "A4_clearing":          "SupplFig3d",
    "A5_magnification":     "SupplFig3e",
    "B_bleaching":          "SupplFig3g",
    "C_detection_depth":    "SupplFig3h",
}


def panel_source_table(order, repl, null):
    """The per-compound replicate r and the null draw behind each box."""
    import pandas as _pd
    rows = []
    for case in order:
        s = repl[case]
        rows.append(_pd.DataFrame({"case": case, "kind": "replicate",
                                   "compound": list(s.index), "r": list(s.values)}))
        rows.append(_pd.DataFrame({"case": case, "kind": "null",
                                   "compound": None, "r": list(null[case])}))
    return _pd.concat(rows, ignore_index=True)

MARGIN = 0.10   # equivalence margin in replicate Pearson r

def p_to_stars(p):
    if np.isnan(p): return 'n.d.'
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'

def align_pairs(a, b):
    # a, b: pd.Series indexed by compound -> complete (non-NaN) common compounds
    idx = a.index.intersection(b.index)
    sub = pd.DataFrame({'a': a.reindex(idx), 'b': b.reindex(idx)}).dropna()
    return sub['a'].values, sub['b'].values, list(sub.index)

def _wilcoxon_one(d):
    # one-sample Wilcoxon signed-rank, H0: median(d)=0 (paired by compound). NaN-safe.
    d = np.asarray(d, float); d = d[~np.isnan(d)]
    if len(d) < 2 or np.allclose(d, 0): return np.nan
    try: return float(wilcoxon(d)[1])
    except ValueError: return np.nan

def gap_paired_delta(ra, rb, na, nb, kind, margin=MARGIN, nboot=2000, seed=42):
    """Confound-robust between-condition post-hoc on the GAP (replicate-r minus that condition's null median):
    Dgap_i = (r_i,A - median(null_A)) - (r_i,B - median(null_B)), paired by compound. Bootstrap CI over
    compounds (equiv 90% / diff 95%); the one-sample signed-rank p of Dgap is a nominal companion."""
    gray_a = float(np.nanmedian(na)); gray_b = float(np.nanmedian(nb))
    av, bv, cm = align_pairs(ra, rb); n = len(av)
    if n < 2:
        return dict(n=n, gap=np.nan, blue=np.nan, lo=np.nan, hi=np.nan, level=None, flag='insufficient n', p=np.nan)
    d_gap = (av - gray_a) - (bv - gray_b); d_blue = av - bv
    level = 90 if kind == 'equivalence' else 95
    loq, hiq = (100 - level) / 2, 100 - (100 - level) / 2
    rng = np.random.default_rng(seed)
    bs = np.array([np.median(d_gap[rng.integers(0, n, n)]) for _ in range(nboot)])
    lo, hi = float(np.percentile(bs, loq)), float(np.percentile(bs, hiq))
    gap = float(np.median(d_gap)); blue = float(np.median(d_blue)); p = _wilcoxon_one(d_gap)
    if kind == 'equivalence':
        if -margin <= lo and hi <= margin: flag = f'WITHIN +/-{margin} (90% CI): supports interchangeability'
        elif lo > margin or hi < -margin:  flag = 'CI beyond margin: difference, not equivalence'
        else: flag = 'CI straddles margin: inconclusive (NOT equivalence)'
    else:
        flag = 'CI excludes 0 (95%): difference' if (lo > 0 or hi < 0) else 'CI includes 0: no detectable difference'
    return dict(n=n, gap=gap, blue=blue, lo=lo, hi=hi, level=level, flag=flag, p=p)

def _verdict(res):
    f = res['flag']
    if f.startswith('WITHIN'): return '≡'
    if ('beyond' in f) or ('excludes 0' in f): return '≠'
    return 'ns'   # equivalence-straddle (inconclusive) or difference-includes-0 (no detectable difference)

def panel_analysis(order, repl, null, labels, title, fname,
                   equiv=(), diff=(), ylim=(-0.3, 1.5), descriptive=False, confounded=(), nrep=None):
    """Boxes = Pearson r (replicate vs null). No omnibus, no mAP, no BH.
    Within-condition: replicate distribution vs null distribution by Mann-Whitney U (direct; raw nominal p; stars).
    Between-condition (all-vs-all, gap-corrected Δ, bootstrap CI over compounds = primary verdict):
    designated `equiv` pairs use a 90% CI vs +/-MARGIN (≡ / inconclusive / ≠); every other pair uses a
    95% CI (excludes 0 = ≠); the gap-corrected signed-rank p is a nominal companion. Brackets show all pairs
    when <=4 conditions, otherwise only the designated equivalence pairs (the rest are in the printout).
    `nrep`: optional dict cond->per-compound replicate counts, shown as 'r=median (min-max)' on the x-axis.
    `confounded` pairs are flagged. (`diff`/`descriptive` accepted for call compatibility.)"""
    rows = []
    for c in order:
        for cc, v in repl[c].dropna().items(): rows.append({'condition': c, 'type': 'Replicates', 'corr': float(v)})
        for v in null[c]: rows.append({'condition': c, 'type': 'Null', 'corr': float(v)})
    plot = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(max(5.5, 1.7 * len(order) + 1), 6))
    sns.boxplot(data=plot, x='condition', y='corr', hue='type', order=order,
                hue_order=['Null', 'Replicates'], palette={'Null': 'dimgrey', 'Replicates': 'steelblue'},
                showfliers=False, width=0.6, ax=ax)
    ax.set_ylabel('Median pairwise Pearson r'); ax.set_xlabel(''); ax.set_ylim(*ylim)
    ax.set_xticks(range(len(order)))
    def _xl(c):
        s = f"{labels.get(c, c)}\nc={repl[c].dropna().shape[0]}"
        if nrep is not None and c in nrep and len(nrep[c]):
            a = np.asarray(nrep[c]); s += f", r={int(np.median(a))} ({int(a.min())}-{int(a.max())})"
        return s
    ax.set_xticklabels([_xl(c) for c in order])
    off = 0.15

    # Within-condition: replicate distribution vs null distribution, Mann-Whitney U (direct; raw nominal p)
    wp = [mannwhitneyu(repl[c].dropna().values, null[c], alternative='two-sided')[1]
          if repl[c].dropna().shape[0] >= 1 else np.nan for c in order]
    y1 = ylim[1] * 0.62
    for i, c in enumerate(order):
        ax.text(i, y1, p_to_stars(wp[i]), ha='center', va='bottom', fontsize=10, fontweight='bold', color='#1a3e8c')

    # Between-condition all-vs-all on gap-corrected Δ; CI is the verdict for every pair
    idx = {c: i for i, c in enumerate(order)}
    equiv_set = set(frozenset(p) for p in equiv)
    conf_set = set(frozenset(p) for p in confounded)
    allpairs = [(order[a], order[b]) for a in range(len(order)) for b in range(a + 1, len(order))]
    res = {}
    for (a, b) in allpairs:
        kind = 'equivalence' if frozenset((a, b)) in equiv_set else 'difference'
        res[(a, b)] = gap_paired_delta(repl[a], repl[b], null[a], null[b], kind); res[(a, b)]['kind'] = kind

    # Brackets: all pairs if <=4 conditions (readable); else ONLY the designated equivalence pairs
    to_draw = allpairs if len(order) <= 4 else [p for p in allpairs if res[p]['kind'] == 'equivalence']
    ybase = ylim[1] * 0.80
    for kk, (a, b) in enumerate(to_draw):
        ia, ib = idx[a], idx[b]; y = ybase + kk * 0.11
        ax.plot([ia + off, ia + off, ib + off, ib + off], [y, y + 0.02, y + 0.02, y], color='#666', lw=0.9)
        ax.text((ia + ib) / 2 + off, y + 0.03, _verdict(res[(a, b)]), ha='center', va='bottom', fontsize=11, color='#333')

    ax.legend(handles=[Patch(facecolor='dimgrey', label='Null'), Patch(facecolor='steelblue', label='Replicates')],
              loc='lower left', fontsize=9)
    ax.set_title(f'{title}\nwithin: replicate-vs-null Mann-Whitney U (nominal; stars) · '
                 f'post-hoc all-vs-all: gap-Δ CI primary (equiv 90% vs +/-{MARGIN} ≡ / diff 95% excl.0 ≠); signed-rank p nominal',
                 fontsize=7.5)
    fig.tight_layout()
    save_panel(fig, FNAME_PANEL[fname],
               data=panel_source_table(order, repl, null),
               caption=title,
               notebook='analysis/3_SupplFigure3/3_Robustness_Combined_Final.ipynb')
    plt.show()

    print(f'=== {title} ===')
    print('Within-condition replicate-vs-null (MWU, nominal):',
          {labels.get(c, c): f'{p_to_stars(wp[i])}({wp[i]:.3g})' for i, c in enumerate(order)})
    klab = {'equivalence': 'EQUIV', 'difference': 'diff '}
    for (a, b) in allpairs:
        r = res[(a, b)]; conf = ' [confounded]' if frozenset((a, b)) in conf_set else ''
        print(f"  {klab[r['kind']]} {labels.get(a, a)} - {labels.get(b, b)}: gapD={r['gap']:+.3f} "
              f"{r['level']}%CI[{r['lo']:+.3f},{r['hi']:+.3f}] (blueD={r['blue']:+.3f}) n={r['n']} "
              f"signed-rank p={r['p']:.3g} (nominal) -> {_verdict(r)}{conf}  | {r['flag']}")
    return plot
print('stat layer ok')

## Panel A — Z-plane subsampling (v1; c=52 compounds, ~4 replicates)
Boxes/null reproduced from `3_PercentReplicating_certain_slices.ipynb` (per-compound replicate-r;
3000-sample null, `random.seed(42)`). Corrected stats on top. Equivalence (±0.10), baseline = full
stack: full − 6-plane (`sparse6`), full − 9-plane (`sparse9`).

In [ ]:
# Panel A — data layer reproduced verbatim (per-compound r + 3000 null), then corrected stats
A_CASES = ['section1_z2','section1_z7','section1_z11','section1_sparse3',
           'section1_sparse6','section1_sparse9','section1_12planes']
A_LBL = {'section1_z2':'z2 (shallow)','section1_z7':'z7 (central)','section1_z11':'z11 (deep)',
         'section1_sparse3':'3-plane','section1_sparse6':'6-plane','section1_sparse9':'9-plane',
         'section1_12planes':'full stack'}
perturbation='Metadata_pert_name'; cmpd_conc='Metadata_conc_step'; cmpd_short_name='Metadata_cmpdname'
A_repl, A_null, A_nrep = {}, {}, {}
for case in A_CASES:
    d = pd.read_parquet(f'{V1_SLICES}/selected_{case}.parquet').dropna(axis='columns', how='all')
    d = d[d['Metadata_pert_type'] == 'trt'].copy()
    d['Metadata_conc_step'] = d.groupby('Metadata_cmpdname')['Metadata_cmpd_conc'].rank(ascending=True, method='dense')
    d['Metadata_pert_name'] = d['Metadata_cmpdname'] + '_' + d['Metadata_conc_step'].astype(str)
    maxstep = d.groupby('Metadata_cmpdname')['Metadata_conc_step'].transform('max')
    conc_df = d[d['Metadata_conc_step'] > maxstep - 1].copy()      # highest conc per compound
    rep_counts = conc_df.groupby(perturbation).size()
    n_rep = int(rep_counts[rep_counts >= 2].median())
    nulls = corr_between_non_replicates(conc_df, 3000, n_rep, perturbation)
    cd = process_correlation_data(conc_df, perturbation, cmpd_short_name, cmpd_conc)
    A_repl[case] = cd.set_index('Metadata_cmpdname')['corr']
    A_null[case] = np.array(nulls)
    A_nrep[case] = conc_df.groupby('Metadata_cmpdname').size().values
panel_analysis(A_CASES, A_repl, A_null, A_LBL,
               'Z-plane subsampling — v1 (c=52)', 'A_zplane_subsampling',
               equiv=[('section1_12planes','section1_sparse6'), ('section1_12planes','section1_sparse9')],
               ylim=(-0.4, 1.6), nrep=A_nrep)

In [ ]:
# ===== Builder for v3 matched panels (reproduces repro_fig matched=True data layer, keeps compound id) =====
def build_v3_matched(conds, with_v1=False):
    dd = {}
    for c in conds:
        d = load_v3(c).copy(); d['cmpd'] = d['Metadata_cmpdname']; dd[c] = d
    order = list(conds)
    if with_v1:
        v1t = load_v1_agg(); v1t = v1t[v1t['Metadata_pert_type'] == 'trt']; parts = []
        for sh, full in V1NAME.items():
            s = v1t[(v1t['Metadata_cmpdname'] == full) & np.isclose(v1t['Metadata_cmpd_conc'].astype(float), V1DOSE[sh])].copy()
            s['cmpd'] = sh; parts.append(s)
        dd['v1'] = pd.concat(parts, ignore_index=True); order = ['v1'] + order
    fs = shared_features(*dd.values())
    cmpds = sorted(set.intersection(*[set(dd[c]['cmpd'].unique()) for c in order]))
    nmatch = {cc: int(min((dd[c]['cmpd'] == cc).sum() for c in order)) for cc in cmpds}
    repl, null = {}, {}
    for c in order:
        d = dd[c].reset_index(drop=True); rng = np.random.default_rng(7); rr = {cc: [] for cc in cmpds}
        for b in range(200):
            sel = np.concatenate([rng.choice(np.where(d['cmpd'].values == cc)[0], nmatch[cc], replace=False) for cc in cmpds])
            sub = d.iloc[sel]; C = np.corrcoef(sub[fs].values); np.fill_diagonal(C, np.nan); lab = sub['cmpd'].values
            for cc in cmpds:
                ii = np.where(lab == cc)[0]
                if len(ii) > 1: rr[cc].append(np.nanmedian(C[np.ix_(ii, ii)]))
        repl[c] = pd.Series({cc: float(np.mean(rr[cc])) for cc in cmpds if rr[cc]})
        null[c] = matched_null_df(d, fs, 'cmpd', 2000, 7)
    return order, repl, null, nmatch
print('v3 matched builder ok')

## Panel A2 — Z-sampling density (v1 5 µm · nyquist 14z 5 µm · nyquist 44z 1.5 µm)
`repro_fig('upsampling', with_v1=True)` data layer. Cross-experiment pairing (v1 vs v3) on the shared
compounds — **caveat: different plates/runs.** Difference: 14z(5 µm) − 44z(1.5 µm). Equivalence:
v1(5 µm) − 14z(5 µm).

In [ ]:
A2_order, A2_repl, A2_null, A2_nm = build_v3_matched(['nyquist_14planes', 'nyquist_all'], with_v1=True)
A2_LBL = {'v1':'5 µm (v1 orig)', 'nyquist_14planes':'5 µm (nyq 14z)', 'nyquist_all':'1.5 µm (nyq 44z)'}
print('A2 matched N/compound:', A2_nm, '| order:', A2_order)
panel_analysis(A2_order, A2_repl, A2_null, A2_LBL,
               'Z-sampling density — v1 / 14z / 44z (cross-experiment; caveat: diff plates)', 'A2_zdensity',
               diff=[('nyquist_14planes','nyquist_all')], equiv=[('v1','nyquist_14planes')], confounded=[('v1','nyquist_all')], nrep={c: list(A2_nm.values()) for c in A2_order})

## Panel A3 — Spheroid size / seeding density (v2: 270 / 540 / 810 cells/well)
Data from v2 seeding-density sections: **section1 = base (270), section2 = 2× (540), section3 = 3×
(810)**. **Formally tested all-vs-all** (gap-corrected paired-Δ, 95% CI primary; signed-rank nominal). Matched
per-compound replicate-r + matched null (seed 7), same data layer as the v3 panels. Caveat: at the
largest size (810) the unstained interiors (A6) can bias replicate-r — interpret the size deltas with
that in mind. **Gemcitabine genuinely dissolves the spheroid → real singleton where it occurs, so it
drops per the missing-data rule.**

In [ ]:
# Panel A3 — v2 seeding density (descriptive): base/2x/3x ~ 270/540/810 cells/well
V2SEC = profiles("exp2_spheroid_size", "sections")
A3_FILES = {'270': f'{V2SEC}/grit_section1_12planes.parquet',
            '540': f'{V2SEC}/grit_section2.parquet',
            '810': f'{V2SEC}/grit_section3.parquet'}
order3 = list(A3_FILES); dfs3 = {}
for k, p in A3_FILES.items():
    d = pd.read_parquet(p).dropna(axis='columns', how='all')
    d = d[d['Metadata_pert_type'] == 'trt'].copy(); d['cmpd'] = d['Metadata_cmpdname']; dfs3[k] = d
fs3 = shared_features(*dfs3.values())
cmpds3 = sorted(set.intersection(*[set(dfs3[k]['cmpd'].unique()) for k in order3]))
nm3 = {cc: int(min((dfs3[k]['cmpd'] == cc).sum() for k in order3)) for cc in cmpds3}
A3_repl, A3_null = {}, {}
for k in order3:
    d = dfs3[k].reset_index(drop=True); rng = np.random.default_rng(7); rr = {cc: [] for cc in cmpds3}
    for b in range(200):
        sel = np.concatenate([rng.choice(np.where(d['cmpd'].values == cc)[0], nm3[cc], replace=False) for cc in cmpds3])
        sub = d.iloc[sel]; C = np.corrcoef(sub[fs3].values); np.fill_diagonal(C, np.nan); lab = sub['cmpd'].values
        for cc in cmpds3:
            ii = np.where(lab == cc)[0]
            if len(ii) > 1: rr[cc].append(np.nanmedian(C[np.ix_(ii, ii)]))
    A3_repl[k] = pd.Series({cc: float(np.mean(rr[cc])) for cc in cmpds3 if rr[cc]})
    A3_null[k] = matched_null_df(d, fs3, 'cmpd', 2000, 7)
print('A3 seeding matched N/compound:', nm3)
panel_analysis(order3, A3_repl, A3_null, {k: f'{k} cells/well' for k in order3},
               'Spheroid size / seeding density — v2', 'A3_spheroid_size',
               diff=[('270', '540'), ('270', '810'), ('540', '810')], nrep={k: list(nm3.values()) for k in order3})

## Panel A4 — Clearing (40/40 % · 20/40 % · none)
Clearing reproducibility (criterion 2a) data layer. Equivalence (CORE, ±0.10): 40/40 (`nyquist_14planes`)
− 20/40 (`20_40`). Difference: 40/40 − none (`noclear`).

In [ ]:
# Panel A4 — reproduces criterion-2a construction (single rng seed 7 across conditions)
REPRO = ['20_40', 'noclear', 'nyquist_14planes']
rep_dfs = {c: load_v3(c) for c in REPRO}
feats2a = shared_features(*rep_dfs.values())
cmpds4 = sorted(set.union(*[set(d['Metadata_cmpdname']) for d in rep_dfs.values()]))
nmatch4 = {c: int(min((d['Metadata_cmpdname'] == c).sum() for d in rep_dfs.values())) for c in cmpds4}
A4_repl, A4_null = {}, {}
_rng_m = np.random.default_rng(7)
for c in REPRO:
    d = rep_dfs[c].reset_index(drop=True); lab = d['Metadata_cmpdname'].values; rr = {cc: [] for cc in cmpds4}
    for _b in range(200):
        sel = np.concatenate([_rng_m.choice(np.where(lab == cc)[0], nmatch4[cc], replace=False) for cc in cmpds4])
        sub = d.iloc[sel]; Cs = np.corrcoef(sub[feats2a].values); np.fill_diagonal(Cs, np.nan); labs = sub['Metadata_cmpdname'].values
        for cc in cmpds4:
            ii = np.where(labs == cc)[0]
            if len(ii) > 1: rr[cc].append(np.nanmedian(Cs[np.ix_(ii, ii)]))
    A4_repl[c] = pd.Series({cc: float(np.mean(rr[cc])) for cc in cmpds4 if rr[cc]})
    A4_null[c] = matched_null_df(d, feats2a, 'Metadata_cmpdname', 2000, 7)
A4_order = ['nyquist_14planes', '20_40', 'noclear']
A4_LBL = {'nyquist_14planes':'40/40 (orig)', '20_40':'20/40', 'noclear':'none'}
print('A4 matched N/compound:', nmatch4)
panel_analysis(A4_order, A4_repl, A4_null, A4_LBL, 'Clearing', 'A4_clearing',
               equiv=[('nyquist_14planes','20_40')], diff=[('nyquist_14planes','noclear')], nrep={c: list(nmatch4.values()) for c in A4_order})

## Panel A5 — Magnification (30× orig · 10× · 40×)
`repro_fig('magnification')` data layer. All-vs-all difference (gap-Δ 95% CI primary; signed-rank
nominal). Key claims: **30× − 40×** (CI excludes 0 — holds) and **10× − 40×** (CI includes 0 — not
robust once paired + gap-corrected); 40× is the failing magnification (not above its own null).

In [ ]:
A5_order, A5_repl, A5_null, A5_nm = build_v3_matched(['nyquist_14planes', '10x', '40x'], with_v1=False)
A5_LBL = {'nyquist_14planes':'30× (orig)', '10x':'10×', '40x':'40×'}
print('A5 matched N/compound:', A5_nm, '| order:', A5_order)
panel_analysis(A5_order, A5_repl, A5_null, A5_LBL, 'Magnification', 'A5_magnification',
               diff=[('10x','40x'), ('nyquist_14planes','40x'), ('nyquist_14planes','10x')], nrep={c: list(A5_nm.values()) for c in A5_order})

## Panel B — Intensity decay / bleaching (nyquist v3 vs v1, 5 channels)
Ported verbatim from `clearing_comparison_stats.ipynb` (criterion 4). Curve = median; shading =
per-depth 95% bootstrap CI across wells; crimson spans = cluster-perm sig depths; **BH-FDR across 5
channels.** Caveat: CI is across-well within 2 (v3)/4 (v1) plates → understates between-experiment
uncertainty.

In [ ]:
# SupplFig3g is the one part of this notebook that needs per-object data; panels a-e
# above come entirely from the deposited section and slice tables. Guarded here rather
# than skipping the whole notebook, which used to cost all seven SupplFig3 panels.
if not (V1_SLICE.is_dir() and V3_SLICE.is_dir()):
    print("SKIP SupplFig3g: needs the per-slice feature dumps (V1_SLICE / V3_SLICE), "
          "~19.5 GB and in no download tier. Panels a-e above are unaffected.")
else:
    COMP = {'HOECHST': 'nuclei', 'MITO': 'cells', 'CONC': 'cells', 'PHAandWGA': 'cells', 'SYTO': 'cells'}
    def slice_perwell(folder, comp_map, img2cond=None, cond=None):
        fs = sorted(glob.glob(f'{folder}/HCT116_Slice*MedianAgg.parquet'),
                    key=lambda p: int(re.search(r'Slice(\d+)', p).group(1)))
        chcols = {ch: f'Intensity_MeanIntensity_{ch}_{comp}' for ch, comp in comp_map.items()}
        perch = {ch: {} for ch in comp_map}
        for f in fs:
            sidx = int(re.search(r'Slice(\d+)', f).group(1)); d = pd.read_parquet(f); avail = d.columns
            d = d[d['Metadata_cmpdname'] == 'dmso']
            if img2cond is not None: d = d[d['Metadata_image_id'].map(img2cond) == cond]
            if len(d) == 0: continue
            d = d.copy(); d['wid'] = d['Metadata_Barcode'].astype(str) + '_' + d['Metadata_Well'].astype(str)
            for ch, cc in chcols.items():
                if cc in avail: perch[ch][sidx] = d.groupby('wid')[cc].median()
        return {ch: pd.DataFrame(perch[ch]).T.sort_index() for ch in comp_map if perch[ch]}
    nyq = slice_perwell(V3_SLICE, COMP, img2cond=IMG2COND, cond='nyquist')
    v1s = slice_perwell(V1_SLICE, COMP)
    def znorm(df):
        if 0 not in df.index: return df.iloc[0:0]
        z0 = df.loc[0]; keep = z0[(z0.notna()) & (z0 > 0)].index; return df[keep].divide(z0[keep], axis=1)
    nyqn = {ch: znorm(nyq[ch]) for ch in nyq}; v1sn = {ch: znorm(v1s[ch]) for ch in v1s}
    GRID = np.arange(0.0, 55.0001, 2.5)
    def interp_curves(dfn, step):
        out = []
        for w in dfn.columns:
            v = dfn[w].values; d = dfn.index.values * step; m = ~np.isnan(v)
            if m.sum() >= 2: out.append(np.interp(GRID, d[m], v[m]))
        return np.array(out)
    rng = np.random.default_rng(42)
    fig, axes = plt.subplots(1, len(CHANNELS), figsize=(3 * len(CHANNELS), 3.4), sharey=True)
    rows, gC = [], []
    for ax, ch in zip(axes, CHANNELS):
        for lab, col, dfn, step in [('v1', '#888', v1sn.get(ch), 5.0), ('nyquist', 'crimson', nyqn.get(ch), 1.5)]:
            if dfn is None or dfn.empty: continue
            um = dfn.index.values * step
            stt = [_boot_ci(dfn.loc[z].values, rng, np.median) for z in dfn.index]
            m = np.array([t[0] for t in stt]); lo = np.array([t[1] for t in stt]); hi = np.array([t[2] for t in stt])
            ax.plot(um, m, marker='o', ms=2, color=col, label=f'{lab} (n={dfn.shape[1]})')
            ax.fill_between(um, lo, hi, color=col, alpha=0.16, lw=0)
        ax.set_title(f'{ch} ({COMP[ch]})'); ax.set_xlabel('depth (µm)')
        if ch in v1sn and ch in nyqn and not v1sn[ch].empty and not nyqn[ch].empty:
            V = interp_curves(v1sn[ch], 5.0); N = interp_curves(nyqn[ch], 1.5)
            bands, gp = cluster_perm(N, V, GRID, rng)        # curve cluster-permutation test (only test)
            rows.append(dict(ch=ch, bands=bands, gp=gp)); gC.append(gp)
        else:
            rows.append(None); gC.append(np.nan)
    axes[0].set_ylabel('intensity (norm. to own z0)'); axes[0].legend(fontsize=7)
    aC = _bh_arr([g for g in gC if not np.isnan(g)])     # BH-FDR across the 5 stains
    it = iter(aC)
    for ax, r in zip(axes, rows):
        if r is None: continue
        ac = next(it); r['aC'] = ac
        if ac < 0.05:                                    # shade only stains significant after BH (shading == reported call)
            for (s, e, pp) in _sig_bands(r['bands']): ax.axvspan(s, e, color='crimson', alpha=0.10, lw=0)
        ax.annotate(f'curve p={ac:.3f} {_stars(ac)}', xy=(0.5, 0.02), xycoords='axes fraction',
                    ha='center', va='bottom', fontsize=7, color='crimson')
    fig.suptitle('Bleaching: nyquist (v3, 1.5 µm) vs v1 (5 µm) — z0-normalized intensity vs depth (DMSO)\n'
                 'median ± 95% bootstrap CI across wells; shaded = depths of significant divergence '
                 '(curve cluster-permutation test; BH-FDR across 5 stains)')
    fig.tight_layout()
    save_panel(fig, 'SupplFig3g',
               data=pd.read_csv(CACHE_3G),
               caption='Channel intensity vs imaging depth, two spheroid sizes',
               notebook='analysis/3_SupplFigure3/3_Robustness_Combined_Final.ipynb')
    plt.show()
    print('=== Panel B: bleaching nyquist vs v1 — curve cluster-permutation (BH across 5 stains) ===')
    for r in rows:
        if r is None: continue
        b = ', '.join(f'[{s:.0f}-{e:.0f}µm p={p:.3f}]' for s, e, p in _sig_bands(r['bands'])) or 'none'
        print(f"  {r['ch']:10s} curve adj p={r['aC']:.4f} {_stars(r['aC'])}  sig {b}")
    print('Caveat: across-well CI within 1 acquisition per condition (n = 58 v1 / 18 nyquist DMSO wells).')


## Z-sampling & bleaching — shown vs inferred

**Shown (two independent results):**
- *Sampling (Panel A, n=52):* reproducibility is preserved out to **12 µm** — the 6-plane (~12 µm) and 9-plane (~7.5 µm) subsamples are equivalent to the 5 µm full stack within the ±0.10 margin; it degrades only by 30 µm. This is an **analysis-level** result (subsamples of one acquisition), **not** a bleaching claim.
- *Dose→bleaching (Panel B):* photobleaching scales with the number of **acquired** planes — the 44-plane acquisition shows greater far-red (mitochondrial) intensity loss at depth than the 13-plane acquisition. Channel-specific (labile far-red/MITO; HOECHST/SYTO stable).

**Inferred (not directly measured):** acquiring **directly at 12 µm (≈6 planes)** rather than 5 µm (13) or 1.5 µm (44) should incur fewer plane-exposures → lower cumulative dose → less bleaching — a monotonic extrapolation of the measured dose-response below the 13-plane point (bleaching ∝ cumulative photon dose), valid **at equal per-plane exposure/laser power**.

**Caveats:** (1) the dose saving holds only if coarser sampling is not offset by longer per-plane exposure or higher power; (2) the bleaching benefit is mainly for labile dyes.

*Suggested text:* "Subsampling to 12 µm preserved replicate reproducibility (equivalent to the 5 µm full stack within a ±0.10 margin; n=52, Fig 3A). Independently, photobleaching scaled with the number of acquired planes — the 44-plane acquisition showed greater far-red (mitochondrial) intensity loss at depth than the 13-plane acquisition (Fig 3B). Acquiring directly at 12 µm (≈6 planes) would therefore be expected to further reduce cumulative light dose and associated photobleaching of labile dyes, at equal per-plane exposure, though this was not measured directly."

## Panel C — Cell detection vs depth (v1 / 20_40 / noclear / nyquist 14z)
Ported from criterion 3. Curve = mean; shading = per-depth 95% bootstrap CI across wells; cluster-perm
vs 20/40 reference. **Now BH-FDR across the 3 condition-pairs** (symmetry with Panel B); raw + adjusted
printed. Same across-well CI caveat.

In [ ]:
# SupplFig3h is the one part of this notebook that needs per-object data; panels a-e
# above come entirely from the deposited section and slice tables. Guarded here rather
# than skipping the whole notebook, which used to cost all seven SupplFig3 panels.
if not (V1_SC.exists() and V3_SC.exists()):
    print("SKIP SupplFig3h: needs the single-cell feature dumps (V1_SC / V3_SC), "
          "~19.5 GB and in no download tier. Panels a-e above are unaffected.")
else:
    sc = pd.read_parquet(V3_SC, columns=['Metadata_image_id','Metadata_cmpdname','Metadata_z','Metadata_Well_nuclei','Metadata_Barcode'])
    sc['cond'] = sc['Metadata_image_id'].map(IMG2COND)
    sc['wid'] = sc['Metadata_Barcode'].astype(str) + '_' + sc['Metadata_Well_nuclei'].astype(str)
    dm = sc[sc['Metadata_cmpdname'] == 'dmso']
    Z14 = list(np.linspace(0, 43, 14).round().astype(int))
    perwell = {}
    for cond in ['20_40', 'noclear', 'nyquist']:
        s = dm[dm['cond'] == cond]
        if cond == 'nyquist': s = s[s['Metadata_z'].isin(Z14)]
        zs = sorted(s['Metadata_z'].unique()); wells = sorted(s['wid'].unique())
        perwell[cond] = s.groupby(['Metadata_z', 'wid']).size().unstack('wid').reindex(index=zs, columns=wells).fillna(0.0)
    v1sc = pd.read_parquet(V1_SC, columns=['Metadata_Site','Metadata_cmpdname','Metadata_Barcode','Metadata_Well','Metadata_cell_line'])
    v1dm = v1sc[(v1sc['Metadata_cell_line'] == 'HCT116') & (v1sc['Metadata_cmpdname'] == 'dmso')].copy()
    v1dm['wid'] = v1dm['Metadata_Barcode'].astype(str) + '_' + v1dm['Metadata_Well'].astype(str)
    zs = sorted(v1dm['Metadata_Site'].unique()); wells = sorted(v1dm['wid'].unique())
    perwell['v1'] = v1dm.groupby(['Metadata_Site', 'wid']).size().unstack('wid').reindex(index=zs, columns=wells).fillna(0.0)
    LBL3 = {'v1':'v1 (orig)', '20_40':'20/40', 'noclear':'no clear', 'nyquist':'nyquist (14z)'}
    COL3 = {'v1':'#888', '20_40':'steelblue', 'noclear':'#d9534f', 'nyquist':'seagreen'}
    STEP3 = {'v1':5.0, '20_40':5.0, 'noclear':5.0, 'nyquist':1.5}
    rng = np.random.default_rng(42)
    fig, ax = plt.subplots(figsize=(7.0, 4.4))
    for cond in ['v1', '20_40', 'noclear', 'nyquist']:
        mat = perwell[cond]; um = mat.index.values * STEP3[cond]
        st = [_boot_ci(mat.loc[z].values, rng, np.mean) for z in mat.index]
        m = np.array([t[0] for t in st]); lo = np.array([t[1] for t in st]); hi = np.array([t[2] for t in st])
        ax.plot(um, m, marker='o', ms=3, color=COL3[cond], label=f'{LBL3[cond]} (n={mat.shape[1]})')
        ax.fill_between(um, lo, hi, color=COL3[cond], alpha=0.16, lw=0)
    ax.set_xlabel('Imaging depth (µm)'); ax.set_ylabel('Cells detected per well per slice')
    GRIDc = np.arange(0.0, 55.0001, 5.0)
    def interp_cond(cond):
        mat = perwell[cond]; d = mat.index.values * STEP3[cond]
        return np.array([np.interp(GRIDc, d, mat[w].values) for w in mat.columns])
    Cc = {c: interp_cond(c) for c in ['v1', '20_40', 'noclear', 'nyquist']}
    ref = '20_40'; B = Cc[ref]
    comps = [('nyquist', 'nyquist (14z)'), ('noclear', 'no clear'), ('v1', 'v1 (orig)')]
    # Curve cluster-permutation test only — each condition vs the 20/40 reference
    results = []
    for cond, lab in comps:
        bands, gp = cluster_perm(Cc[cond], B, GRIDc, rng)
        results.append(dict(cond=cond, lab=lab, bands=bands, gp=gp))
    gp_adj = _bh_arr([r['gp'] for r in results])         # BH-FDR across the 3 condition-pairs
    for r, ga in zip(results, gp_adj): r['gp_adj'] = ga
    for k, r in enumerate(results):
        yf = 0.985 - 0.045 * k
        if r['gp_adj'] < 0.05:                            # draw bars only where the BH-adjusted curve test is significant
            for (snm, enm, pp) in _sig_bands(r['bands']):
                ax.plot([snm, enm], [yf, yf], transform=ax.get_xaxis_transform(),
                        color=COL3[r['cond']], lw=4, solid_capstyle='butt', clip_on=False)
    ax.legend(fontsize=8, loc='lower right')
    ax.set_title('Cell detection vs depth (DMSO) — v1 + v3\n'
                 'mean ± 95% bootstrap CI across wells; bars = depths differing from 20/40\n'
                 '(curve cluster-permutation test; BH-FDR across 3 condition-pairs)')
    fig.tight_layout()
    save_panel(fig, 'SupplFig3h',
               data=pd.read_csv(CACHE_3H),
               caption='Cell detection vs imaging depth by clearing condition',
               notebook='analysis/3_SupplFigure3/3_Robustness_Combined_Final.ipynb')
    plt.show()
    print('=== Panel C: cell detection vs depth — curve cluster-permutation vs 20/40 (BH-FDR across 3 pairs) ===')
    for r in results:
        b = ', '.join(f'[{s:.0f}-{e:.0f}µm p={p:.3f}]' for s, e, p in _sig_bands(r['bands'])) or 'none'
        print(f"  {r['lab']:13s} curve raw p={r['gp']:.4f} adj p={r['gp_adj']:.4f} {_stars(r['gp_adj'])}  sig {b}")
    print('Caveat: across-well CI within 2 (v3) / 4 (v1) plates; n wells = ' + ', '.join(f'{LBL3[c]} {perwell[c].shape[1]}' for c in ['v1','20_40','noclear','nyquist']))
